## load_realtor
Loads the two Realtor.com metro CSVs — **snapshot** (current month) and **history** (full backfill), identical 47-column schema — from `RAW_REALTOR` (`/Volumes/{CATALOG}/raw/realtor/`) into the all-STRING Bronze table `{BRONZE}.realtor_metro_monthly`.

**Write strategy (A):** MERGE on `(cbsa_code, month_date_yyyymm)` with a `row_hash` change guard. The two files **overlap** on the latest month, so a single combined MERGE would raise "multiple source rows matched a target row". Instead each file is MERGEd **sequentially, history before snapshot**, so the snapshot's value wins the overlap month and source keys stay unique per MERGE.

`HouseholdRank` is renamed to `household_rank` via positional schema mapping (header=true + explicit StructType skips row 1 and applies the schema by position). **No archiving** (rolling snapshots; §18 deviation from §10 cell 7). DDL: `libs/ddl/bronze_ddl.py`.

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injected: BRONZE, AUDIT, RAW_REALTOR, PIPELINE_RUN_ID, STATUS_*, StepLog,
# Utils, ingestion_log_insert, spark, dbutils, F, StructType/StructField/StringType.

STEP_SEQUENCE = 1                                   # position is owned by the orchestrator
SOURCE_SYSTEM = "realtor"
SOURCE_PATH   = RAW_REALTOR                         # /Volumes/{CATALOG}/raw/realtor/
TARGET_TABLE  = f"{BRONZE}.realtor_metro_monthly"

# Exact source header (47 cols), file order — used to validate each file's header.
EXPECTED_SOURCE_COLS = [
    "month_date_yyyymm", "cbsa_code", "cbsa_title", "HouseholdRank",
    "median_listing_price", "median_listing_price_mm", "median_listing_price_yy",
    "active_listing_count", "active_listing_count_mm", "active_listing_count_yy",
    "median_days_on_market", "median_days_on_market_mm", "median_days_on_market_yy",
    "new_listing_count", "new_listing_count_mm", "new_listing_count_yy",
    "price_increased_count", "price_increased_count_mm", "price_increased_count_yy",
    "price_increased_share", "price_increased_share_mm", "price_increased_share_yy",
    "price_reduced_count", "price_reduced_count_mm", "price_reduced_count_yy",
    "price_reduced_share", "price_reduced_share_mm", "price_reduced_share_yy",
    "pending_listing_count", "pending_listing_count_mm", "pending_listing_count_yy",
    "median_listing_price_per_square_foot", "median_listing_price_per_square_foot_mm",
    "median_listing_price_per_square_foot_yy",
    "median_square_feet", "median_square_feet_mm", "median_square_feet_yy",
    "average_listing_price", "average_listing_price_mm", "average_listing_price_yy",
    "total_listing_count", "total_listing_count_mm", "total_listing_count_yy",
    "pending_ratio", "pending_ratio_mm", "pending_ratio_yy", "quality_flag",
]
# Target column names: identical except HouseholdRank -> household_rank. The read schema is
# built from these (positional), so the rename happens at read with no extra select.
TARGET_COLS = ["household_rank" if c == "HouseholdRank" else c for c in EXPECTED_SOURCE_COLS]
read_schema = StructType([StructField(c, StringType(), True) for c in TARGET_COLS])

MERGE_KEYS = ["cbsa_code", "month_date_yyyymm"]
# row_hash covers all 47 payload columns -> MERGE skips no-op updates on the overlap month.
HASH_COLS  = TARGET_COLS

In [ ]:
# Open the pipeline_step_log row (RUNNING). Closed explicitly in the work cell (succeed) or
# by step.fail(e) in the 2-line handler.
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "bronze",
    target_table    = TARGET_TABLE,
)
print(f"load_realtor: step_log_id={step.step_log_id}")

In [ ]:
# Per-file header validation before any read. Sort files so 'history' precedes 'snapshot'
# (alphabetical) -> snapshot is MERGEd last and wins the overlap month. No-files CHECK
# inside the try; EXIT outside (dbutils.notebook.exit raises an ordinary exception that
# except Exception would swallow) — CLAUDE.md §10.1.
no_files = False
try:
    files = sorted(f.path for f in dbutils.fs.ls(SOURCE_PATH) if f.path.lower().endswith(".csv"))
    no_files = not files
    if not no_files:
        bad_files = []
        for file_path in files:
            actual = (
                spark.read.format("csv").option("header", "true")
                .load(file_path).limit(0).columns
            )
            if actual != EXPECTED_SOURCE_COLS:
                bad_files.append((file_path, actual))
        if bad_files:
            raise ValueError(
                f"[{TARGET_TABLE}] Header mismatch in {len(bad_files)} file(s).\n"
                f"Expected: {EXPECTED_SOURCE_COLS}\n"
                + "\n".join(f"  {p}\n    actual: {h}" for p, h in bad_files)
            )
        print(f"load_realtor: {len(files)} file(s) passed header validation: {files}")
except Exception as e:
    step.fail(e); raise

if no_files:
    step.no_files()
    dbutils.notebook.exit(f"No CSV files found at {SOURCE_PATH}")

In [ ]:
# Read + shape + MERGE each file in turn (history then snapshot). Per-file MERGE keeps
# source keys unique; sequential order lets snapshot overwrite the overlap month.
try:
    on_clause = " AND ".join(f"t.{k} = s.{k}" for k in MERGE_KEYS)
    total_read = total_inserted = total_updated = 0

    for file_path in files:
        shaped_df = (
            spark.read.format("csv").option("header", "true").option("delimiter", ",")
                .schema(read_schema).load(file_path)
                .withColumn("row_hash", F.md5(F.concat_ws("|", *[F.col(c) for c in HASH_COLS])))
                .withColumn("source_file_path", F.col("_metadata.file_path"))
                .withColumn("inserted_ts", F.current_timestamp())
                .withColumn("run_id", F.lit(PIPELINE_RUN_ID))
        )
        n_read = shaped_df.count()
        shaped_df.createOrReplaceTempView("realtor_staging")

        pre_count = spark.table(TARGET_TABLE).count()
        metrics = spark.sql(f"""
            MERGE INTO {TARGET_TABLE} AS t
            USING realtor_staging AS s
            ON {on_clause}
            WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """).first().asDict()
        post_count = spark.table(TARGET_TABLE).count()

        inserted = metrics.get("num_inserted_rows")
        if inserted is None:
            inserted = post_count - pre_count
        if post_count - pre_count != inserted:
            raise AssertionError(
                f"[{TARGET_TABLE}] Insert-count mismatch for {file_path}: MERGE reported "
                f"{inserted:,} inserts, row count grew by {post_count - pre_count:,}."
            )
        total_read     += n_read
        total_inserted += inserted
        total_updated  += metrics.get("num_updated_rows") or 0
        print(f"load_realtor: {file_path}: read={n_read:,} inserted={inserted:,} "
              f"updated={metrics.get('num_updated_rows')}")

    step.rows_read    = total_read
    step.rows_written = total_inserted
    step.succeed()
    print(f"load_realtor: DONE read={total_read:,} inserted={total_inserted:,} "
          f"updated={total_updated:,}")
except Exception as e:
    step.fail(e); raise

# ingestion_log (leaf tier) after succeed(), outside the try — one row per ingested file.
files_df = spark.createDataFrame([(p,) for p in files], "source_file_path string")
res = ingestion_log_insert(
    spark, AUDIT, files_df, PIPELINE_RUN_ID, step.step_log_id,
    source_system=SOURCE_SYSTEM, target_table=TARGET_TABLE,
)
if res["status"] != STATUS_SUCCEEDED:
    print(f"load_realtor: WARNING ingestion_log insert failed: {res['error_message']}")